In [ ]:
import math
import numpy as np
from typing import Union, NamedTuple, Literal
import scipy

# European Option - Binomial Tree Model

In [ ]:
def EuroPricerBinomial(S: float, K: float, sigma: float, r: float, q: float, T: float, N: float,
                 option_type: Literal["call", "put"] = "call"
                 ):
  """
  European option pricer using a binomial step model.

  Args:
    S: Spot price
    K: Strike price
    sigma: Volatility (annualized)
    r: Risk-free rate
    q: Dividend yield
    T: Time to maturity (years)
    N: Number of time steps
    option_type: "call" or "put"

  Returns:
    EurPricerResult: price of the option
  """
  # Time per step
  dT = T / N

  # up/down factors
  u = np.exp(sigma * np.sqrt(dT))
  d = 1 / u

  # Risk-neutral probability
  p = (np.exp((r - q) * dT) - d) / (u - d)
  # --- Input validation ---
  if not (0 < p < 1):
    raise ValueError("Risk-neutral probability outside [0,1]. Check parameters.")

  # Discount factor
  discFact = np.exp(-r * dT)

  # --- Step 1: Stock prices at maturity ---
  j = np.arange(N + 1)
  S_T = S * (u ** j) * (d ** (N - j))

  # --- Step 2: Payoff at maturity (intrinsic value) ---
  if option_type == "call":
    option_value = np.maximum(S_T - K, 0)
  else:
    option_value = np.maximum(K - S_T, 0)

  # --- Step 3: Backward induction (NO EARLY EXERCISE) ---
  for i in range(N - 1, -1, -1):
    # Continuation value (discounted expected value of holding)
    # For European options, this is the ONLY value at each node
    option_value = discFact * (p * option_value[1:i+2] + (1 - p) * option_value[:i+1])

  return option_value[0]

# Example usage
# eur_call = EuroPricerBinomial(S=100, K=105, sigma=0.2, r=0.05, q=0.0, T=5, N=200, option_type="put")
# print(f"European Call: {eur_call:.4f}")

# European Option - Black-Scholes Merton Model and Greeks

In [ ]:
Number = Union[float, np.ndarray]

def norm_cdf(x: Number) -> Number:
  return 0.5 * (1.0 + scipy.special.erf(x / math.sqrt(2.0)))

def norm_pdf(x: Number) -> Number:
  return np.exp(-0.5 * -x**2) / math.sqrt(2 * math.pi)

class EuroBSPricerCF(NamedTuple):
  price: float
  delta: float
  gamma: float
  vega: float
  theta: float
  rho: float

def EuroBSPricerCF(S: float, K: float, sigma: float, r: float, q:float, T: float,
                   option_type: Literal["call", "put"] = "call"
                   ):

  """
  European option pricer using Black-Scholes closed-form with Greeks.

  Args:
    S: Spot price
    K: Strike price
    sigma: Volatility (annualized)
    r: Risk-free rate
    q: Dividend yield
    T: Time to maturity (years)
    option_type: "call" or "put"

  Returns:
    EuroBSPricerCF: price, delta, gamma, vega, theta, rho
  """

  # --- Convert inputs to numpy arrays for vectorized operations ---
  S = np.asarray(S)
  K = np.asarray(K)
  sigma = np.asarray(sigma)
  T = np.asarray(T)

  # --- Input validation ---
  if np.any(S <= 0) or np.any(K <= 0):
    raise ValueError("S and K must be > 0")
  if np.any(sigma <= 0):
    raise ValueError("sigma must be > 0")
  if np.any(T <= 0):
    raise ValueError("Time to maturity must be > 0")

  # --- Handle expiration (T=0) ---
  if np.all(T == 0):
    if option_type == "call":
      intrinsic = np.maximum(S - K, 0)
      delta = 1.0

    elif option_type == "put":
      intrinsic = np.maximum(K - S, 0)
      delta = -1.0

    return EuroBSPricerCF(
      price = float(intrinsic),
      delta = float(delta),
      gamma = 0.0, vega = 0.0, theta = 0.0, rho = 0.0
    )

  # --- Precompute common terms ---
  sqrt_T = np.sqrt(T)
  discount = np.exp(-r * T)
  div_yield = np.exp(-q * T)

  # --- Compute d1 and d2 (standardized log-moneyness terms) ---
  d_1 = (np.log(S / K)+(r - q + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
  d_2 = d_1 - sigma * sqrt_T

  # --- Compute price and Greeks ---
  if option_type == "call":
    px = S * div_yield * norm_cdf(d_1) - K * discount * norm_cdf(d_2)
    delta = norm_cdf(d_1)
    rho = K * T * discount * norm_cdf(d_2)
    theta = (
        - (S * norm_pdf(d_1) * sigma) / (2 * sqrt_T)
        - r * K * discount * norm_cdf(d_2)
    )

  else:
    px = K * discount * norm_cdf(-d_2) - S * div_yield * norm_cdf(-d_1)
    delta = norm_cdf(d_1) - 1.0
    rho = -K * T * discount * norm_cdf(-d_2)
    theta = (
        -(S * norm_pdf(d_1) * sigma) / (2 * sqrt_T)
        + r * K * discount * norm_cdf(-d_2)
    )

  gamma = norm_pdf(d_1) / (S * sigma * sqrt_T)
  vega  = S * norm_pdf(d_1) * sqrt_T

  return EuroBSPricerOutput(
      price = round(float(px), 4),
      delta = float(delta),
      gamma = float(gamma),
      vega  = float(vega),
      theta = float(theta),
      rho   = float(rho)
  )

# Example usage
option = EuroBSPricerCF(S=100, K=105, sigma=0.2, r=0.05, q=0.0, T=5, option_type="call")
print(f"Option Price: {option.price:.4f}")


Option Price: 26.7618
